In [ ]:
#1、寻找每一行的最大值
logit_maxes = logits.max(1,keepdim=True).value

#2、减去最大值
norm_logits = logits - logit_maxes

#3、求指数e^x，将得分化为正数
counts = norm_logits.exp()

#4、求和，得到分母
counts_sum = counts.sum(1,keepdim=True)

#5、取倒数
counts_sum_inv = counts_sum**-1

#6、计算最终概率softmax
probs = counts *counts_sum_inv

#7、求自然对数
logprobs = probs.log()

#8、抽出正确答案对饮的log概率，求均值并取负号NNL
loss = -logprobs[range(n),Yb].mean()


In [ ]:
#反向推导
#Step8 loss->logprobs
#loss = -logprobs[range(n),Yb].mean()


In [ ]:
#1、求均值
bnmeani = 1/n*hprebn.sum(0,keepdim=True)

#2、减去均值得到偏差
bndiff = hprebn - bnmean

#3、偏差的平方
bndiff2 = bndiff**2

#4、求方差
bnvar = 1/(n-1)*(bndiff2).sum(0,keepdim=True)

#5、求标准差的倒数
bnvar_inv = bnvar*-1

#6、归一化
bnraw = bndiff*bnvar_inv

#7、仿射变换，放缩+平移
hpreact = bngain*bnraw + bnbias


In [1]:
#完整流程
import torch
import matplotlib.pyplot as plt
import torch.nn.functional as F
import random

%matplotlib inline

words = open("names.txt",mode='r').read().splitlines()

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

block_size = 3

def build_dataset(words):
    X,Y = [],[]

    for w in words:
        context = [0]*block_size
        for ch in w+'.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(f"X.shape{X.shape}")
    print(f"Y.shape{Y.shape}")

    return X,Y

random.seed(42)
random.shuffle(words)

n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr,Ytr = build_dataset(words[:n1])
Xval,Yval = build_dataset(words[n1:n2])
Xte,Yte = build_dataset(words[n2:])



X.shapetorch.Size([182625, 3])
Y.shapetorch.Size([182625])
X.shapetorch.Size([22655, 3])
Y.shapetorch.Size([22655])
X.shapetorch.Size([22866, 3])
Y.shapetorch.Size([22866])


In [9]:
n_embd= 10
n_hidden = 200

g = torch.Generator().manual_seed(2147483547)
C = torch.randn((vocab_size,n_embd))
W1 = torch.randn((n_embd*block_size,n_hidden),generator=g)*(5/3)/((n_embd*block_size)**0.5)
b1 = torch.randn(n_hidden,generator=g)*0.1
W2 = torch.randn((n_hidden,vocab_size),generator=g)*0.1
b2 = torch.randn(vocab_size,generator=g)*0.1

bngain = torch.ones((1, n_hidden))
bnbias = torch.zeros((1, n_hidden))

parameters = [C,W1,b1,W2,b2,bngain,bnbias]
for p in parameters:
    p.requires_grad = True

max_steps = 20000
batch_size = 32
n = batch_size
lossi = []

for i in range(max_steps):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix]

    emb = C[Xb]
    embcat = emb.view(emb.shape[0], -1)

    hprebn = embcat @ W1 + b1

    bnmean = hprebn.mean(0, keepdim=True)
    bnvar = hprebn.var(0, keepdim=True, unbiased=True)
    bnvar_inv = (bnvar + 1e-5) ** -0.5
    bnraw = (hprebn - bnmean) * bnvar_inv
    hpreact = bngain * bnraw + bnbias

    h = torch.tanh(hpreact)

    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    dlogits = F.softmax(logits, 1)
    dlogits[range(n), Yb] -= 1  # 这里是减1
    dlogits /= n

    dh = dlogits @ W2.T
    dW2 = h.T @ dlogits
    db2 = dlogits.sum(0)

    dhpreact = (1.0 - h ** 2) * dh

    dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
    dbnbias = dhpreact.sum(0, keepdim=True)
    dhprebn = bngain * bnvar_inv / n * (n * dhpreact - dhpreact.sum(0) - n / (n - 1) * bnraw * (dhpreact * bnraw).sum(0))

    dembcat = dhprebn @ W1.T
    dW1 = embcat.T @ dhprebn
    db1 = dhprebn.sum(0)

    demb = dembcat.view(emb.shape)
    dC = torch.zeros_like(C)
    for k in range(Xb.shape[0]):
        for j in range(Xb.shape[1]):
            ix_char = Xb[k, j]
            dC[ix_char] += demb[k, j]

    grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

    # 梯度裁剪
    for grad in grads:
        grad.clamp_(-5, 5)

    lr = 0.01 if i < 10000 else 0.001
    for p, grad in zip(parameters, grads):
        p.data += -lr * grad

    if i % 1000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())


      0/  20000: 3.7771
   1000/  20000: 2.6066
   2000/  20000: 2.1830
   3000/  20000: 2.7020
   4000/  20000: 2.3635
   5000/  20000: 2.5131
   6000/  20000: 2.5723
   7000/  20000: 2.5959
   8000/  20000: 2.1862
   9000/  20000: 2.3779
  10000/  20000: 2.2732
  11000/  20000: 2.4071
  12000/  20000: 2.3890
  13000/  20000: 2.4695
  14000/  20000: 2.1731
  15000/  20000: 2.5645
  16000/  20000: 2.4350
  17000/  20000: 2.1589
  18000/  20000: 2.4144
  19000/  20000: 2.2841


In [10]:
with torch.no_grad():
  emb = C[Xtr]
  embcat = emb.view(emb.shape[0], -1)
  hpreact = embcat @ W1 + b1
  
  # 在整个训练集上测量真实的均值和方差
  bnmean_global = hpreact.mean(0, keepdim=True)
  bnvar_global = hpreact.var(0, keepdim=True, unbiased=True)

# -------------------------------------------------
# 见证奇迹：让模型自己起名字
g = torch.Generator().manual_seed(2147483647 + 10)

print("\n--- Generated Names ---")
for _ in range(10):
    out = []
    context = [0] * block_size 
    while True:
      # 前向推理
      emb = C[torch.tensor([context])] 
      embcat = emb.view(emb.shape[0], -1) 
      
      # 使用全局统计量进行推理
      hpreact = embcat @ W1 + b1
      hpreact = bngain * (hpreact - bnmean_global) * (bnvar_global + 1e-5)**-0.5 + bnbias
      
      h = torch.tanh(hpreact) 
      logits = h @ W2 + b2 
      
      # 采样
      probs = F.softmax(logits, dim=1)
      ix = torch.multinomial(probs, num_samples=1, generator=g).item()
      
      context = context[1:] + [ix]
      out.append(ix)
      if ix == 0:
        break
    
    print(''.join(itos[i] for i in out))


--- Generated Names ---
carmah.
ami.
hari.
kimle.
revty.
salans.
ejanonen.
deliyha.
kaqei.
nelenia.
